In [ ]:
import pandas as pd

# 1. Load the "Real-Life" data
real_data = pd.read_csv("REAL_DATA.csv")

# Keep a pristine copy of the original dataframe to append our predictions later
df_real = real_data.copy()


# ==========================================
# 2. DATA CLEANING & FEATURE ENGINEERING
# ==========================================

# a. Drop the 'id' column as it has no predictive value
X_real = df_real.drop('id', axis=1)

# b. Handle dates with potentially mixed formats (Carlos's trap)
# 'errors="coerce"' will handle any unparseable garbage text by turning it into NaT (Not a Time)
X_real['date'] = pd.to_datetime(X_real['date'], format='mixed', errors='coerce')
X_real['year'] = X_real['date'].dt.year
X_real['month'] = X_real['date'].dt.month
X_real['day'] = X_real['date'].dt.day

# Drop the original 'date' column
X_real = X_real.drop('date', axis=1)

# c. Handle hidden text in numeric columns (if you identified any during training)
# Example (uncomment and adapt if needed):
# X_real['nb_customers_on_day'] = pd.to_numeric(X_real['nb_customers_on_day'], errors='coerce')

# d. Handle missing values (NaN) WITHOUT dropping any rows!
# Dropping rows would ruin the final competition submission. We fill holes with 0.
X_real = X_real.fillna(0)

# e. Convert categorical variables into dummy/indicator variables (One-Hot Encoding)
X_real = pd.get_dummies(X_real, columns=['state_holiday'])

# f. CRUCIAL STEP: Align columns with the training dataset
# This ensures X_real has the exact same columns as X_train, filling any missing ones with 0.
# (Assumes 'X_train' is already defined in your notebook from the training phase)
X_real = X_real.reindex(columns=X_train.columns, fill_value=0)


# ==========================================
# 3. PREDICTION WITH THE BEST MODEL
# ==========================================

# Select your winning model from the previous tests (e.g., Random Forest)
# (Assumes 'models' dictionary is already defined in your notebook)
best_model = models["Random Forest"] 

# Generate sales predictions for the real-life data
sales_predictions = best_model.predict(X_real)


# ==========================================
# 4. APPEND 'SALES' COLUMN AND EXPORT
# ==========================================

# Add the predicted sales as a new column to the original dataset
real_data['sales'] = sales_predictions

# Round the sales to the nearest integer (since we cannot sell a fraction of an item)
real_data['sales'] = real_data['sales'].round().astype(int)

# Export the final result to a CSV file without the index
# IMPORTANT: Replace "Group1" with your actual group name!
real_data.to_csv("Group1.csv", index=False)

print("File Group1.csv successfully created! The 'sales' column has been added.")